In [36]:
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision
import re
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset,DataLoader
import torch.optim as optim

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\SP23-BCS-
[nltk_data]     160.CUIATD\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\SP23-BCS-
[nltk_data]     160.CUIATD\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\SP23-BCS-
[nltk_data]     160.CUIATD\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [37]:

df=pd.read_csv("IMDB Dataset.csv")

In [38]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [39]:
df.shape

(50000, 2)

In [40]:
df.drop_duplicates(inplace=True)


In [41]:
df.shape
df["review"]=df["review"].str.lower()

In [42]:
#Preprocessing
#This function using regex to remove the Urls from review
#Below Functions using regex function re.sub(pattern,replace,text)
def remove_urls(text):
    text=re.sub(r"http\S+","",text)
    return text

df["review"]=df["review"].apply(remove_urls)

In [43]:
#This function is basically using for removing Punctuation
def remove_punctuation(text):
    text=re.sub(r"[^A-Za-z0-9\s]","",text)
    return text
df["review"]=df["review"].apply(remove_punctuation)

In [44]:
def remove_htmls(text):
    text=re.sub(r"<.*?>","",text)
    return text
df["review"]=df['review'].apply(remove_htmls)

In [45]:
def remove_stopwords(text):
    tokens=word_tokenize(text)
    stop_words=stopwords.words("english")
    
    for word in tokens:
        if word  in stop_words:
            text=text.replace(word,"")
    return text

df["review"]=df["review"].apply(remove_stopwords)

In [46]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


In [47]:
#In this step we will perform Stemming
def steeming(text):
    ps=PorterStemmer()
    stemed_words=[]
    tokens=word_tokenize(text)
    for token in  tokens:
        stemmed_token=ps.stem(token)
        stemed_words.append(stemmed_token)
    return " ".join(stemed_words)

df["review"]=df["review"].apply(steeming)

In [48]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


In [49]:
le=LabelEncoder()
df["sentiment"]=le.fit_transform(df["sentiment"])
y=df["sentiment"]

In [50]:
#Vectorization
tf=TfidfVectorizer(max_features=5000)

X=tf.fit_transform(df["review"])

In [51]:
print(X)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4057140 stored elements and shape (49582, 5000)>
  Coords	Values
  (0, 3538)	0.05515484681268887
  (0, 2868)	0.09361182809242374
  (0, 4940)	0.11467310366614668
  (0, 3002)	0.47200539890110405
  (0, 1275)	0.1357698919196183
  (0, 2289)	0.049500237517476293
  (0, 1933)	0.0791260308382
  (0, 3550)	0.0963974330192639
  (0, 1362)	0.06162489377343992
  (0, 1963)	0.061560444992697486
  (0, 219)	0.08588920995304898
  (0, 1620)	0.0738170550485134
  (0, 4369)	0.041994187696759305
  (0, 4171)	0.17799685402440263
  (0, 3693)	0.033532198172897175
  (0, 4737)	0.26798942924092045
  (0, 3805)	0.04427609784380831
  (0, 4769)	0.05877405881441711
  (0, 1739)	0.037520883911174724
  (0, 4497)	0.07614066339174266
  (0, 3857)	0.17537900435282314
  (0, 1630)	0.06142445471882175
  (0, 1862)	0.07433134577032253
  (0, 3329)	0.06406818508428483
  (0, 3332)	0.0844754682576354
  :	:
  (49581, 4890)	0.10682334916138103
  (49581, 1542)	0.17584072573791829

In [52]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42)

In [53]:
X_train=X_train.toarray()
X_test=X_test.toarray()

In [54]:
trainset=TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float(),
)

testset=TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)


In [59]:
trainloader=DataLoader(trainset,shuffle=True,batch_size=64)
testloader=DataLoader(testset,shuffle=True,batch_size=64)

In [63]:
#Now Building RNN Architecture
class RNN(nn.Module):
    def __init__(self,input_size,hidden_size=128,num_layers=1):
        super().__init__()

        self.hidden_size=hidden_size
        self.num_layers=num_layers
        #RNN Layers
        self.rnn=nn.RNN(input_size,hidden_size,num_layers,batch_first=True)
        #Fully Connected Layer
        self.fc=nn.Linear(hidden_size,1)
    
    def forward(self,x):
        
        h0=torch.zeros(self.num_layers,x.size(0),self.hidden_size)
        out, _=self.rnn(x,h0)
        out=self.fc(out[:,-1,:])
        return out



In [64]:
input_size=X_train.shape[1]
model=RNN(input_size)
criterion=nn.BCELoss()
optimizer=optim.Adam(model.parameters())


In [65]:
#Now the next Step is to Train our RNN model
epochs=10
for epoch in range(epochs):
    model.train()

    for xb,yb in trainloader:
        optimizer.zero_grad()
        xb=xb.unsqueeze(1) #This will Add one additional dimension because our model expect 3D
        output=model(xb)
        
        output=torch.sigmoid(output.squeeze())
        loss=criterion(output,yb)

        loss.backward()
        optimizer.step()
    print(f"For epcoh {epoch}/{epochs} training loss is: {loss.item()}")
        

For epcoh 0/10 training loss is: 0.25478264689445496
For epcoh 1/10 training loss is: 0.16104036569595337
For epcoh 2/10 training loss is: 0.17373915016651154
For epcoh 3/10 training loss is: 0.011218949221074581
For epcoh 4/10 training loss is: 0.027063725516200066
For epcoh 5/10 training loss is: 0.0014259315794333816
For epcoh 6/10 training loss is: 0.014297205954790115
For epcoh 7/10 training loss is: 0.11931943893432617
For epcoh 8/10 training loss is: 0.01605871319770813
For epcoh 9/10 training loss is: 0.0105620501562953


In [67]:
#Model Evaluation
with torch.no_grad():
    correct=0
    total=0

    for xb,yb in testloader:
        xb=xb.unsqueeze(1)
        output=model(xb)
        predicted=(torch.sigmoid(output.squeeze())>0.5).float()
        correct+=(predicted==yb).sum().item()
        total+=yb.size(0)
    print(f"Accuracy is: {correct/total*100}%")


Accuracy is: 85.24720405793559%
